In [0]:
use catalog classic_workspace;
use schema default;

In [0]:
CREATE OR REPLACE TABLE sensor_readings (
  sensor_id     STRING,
  reading_value INT,
  recorded_at   TIMESTAMP
)
USING DELTA
COMMENT 'IoT sensor readings — type widening demo';

In [0]:
describe table sensor_readings;

In [0]:
INSERT INTO sensor_readings VALUES
  ('S001', 1842,  '2024-07-01 09:00:00'),
  ('S002', 2103,  '2024-07-01 09:01:00'),
  ('S003', 987,   '2024-07-01 09:02:00'),
  ('S004', 31500, '2024-07-01 09:03:00');

In [0]:
describe table sensor_readings;

In [0]:
select * from sensor_readings;

In [0]:
%python
from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, StringType, LongType, TimestampType
from datetime import datetime

schema_long = StructType([
    StructField('sensor_id',     StringType(),    False),
    StructField('reading_value', LongType(),      False),
    StructField('recorded_at',   TimestampType(), False),
])

long_readings = [
    Row(sensor_id='S001',
        reading_value=3_200_000_000,          # exceeds INT max (2,147,483,647)
        recorded_at=datetime(2024, 7, 1, 10, 0, 0)),
    Row(sensor_id='S005',
        reading_value=4_100_000_000,
        recorded_at=datetime(2024, 7, 1, 10, 1, 0)),
]

df_long = spark.createDataFrame(long_readings, schema_long)

# This will FAIL — type widening is not yet enabled
df_long.write \
    .format('delta') \
    .mode('append') \
    .option('mergeSchema', 'true') \
    .saveAsTable('sensor_readings')

In [0]:
ALTER TABLE sensor_readings
SET TBLPROPERTIES ('delta.enableTypeWidening' = 'true');

In [0]:
%python
df_long.write \
    .format('delta') \
    .mode('append') \
    .option('mergeSchema', 'true') \
    .saveAsTable('sensor_readings')

display(spark.table('sensor_readings').orderBy('sensor_id', 'recorded_at'))

In [0]:
select * from sensor_readings;

In [0]:
%python
from pyspark.sql.types import DoubleType

schema_double = StructType([
    StructField('sensor_id',     StringType(),    False),
    StructField('reading_value', DoubleType(),    False),  # DOUBLE — wider than LONG but outside the family
    StructField('recorded_at',   TimestampType(), False),
])

double_readings = [
    Row(sensor_id='S006',
        reading_value=1842.75,                 # fractional precision
        recorded_at=datetime(2024, 7, 2, 9, 0, 0)),
]

df_double = spark.createDataFrame(double_readings, schema_double)

df_double.write \
    .format('delta') \
    .mode('append') \
    .option('mergeSchema', 'true') \
    .saveAsTable('sensor_readings')

In [0]:
select * from sensor_readings;

In [0]:
describe table sensor_readings;

In [0]:
CREATE OR REPLACE TABLE iot_events (
  event_id    INT,
  received_at TIMESTAMP,
  payload     VARIANT
)
USING DELTA
COMMENT 'IoT events — variable-schema JSON stored as VARIANT';

In [0]:
INSERT INTO iot_events VALUES
  (1, '2024-07-01 09:00:00',
   PARSE_JSON('{"sensor_id":"S001","temp":23.4,"unit":"C","battery":0.92}')),
  (2, '2024-07-01 09:01:00',
   PARSE_JSON('{"sensor_id":"S002","temp":71.2,"unit":"F"}')),
  (3, '2024-07-01 09:02:00',
   PARSE_JSON('{"sensor_id":"S003","pressure":1013.2,"unit":"hPa","location":{"lat":51.5,"lon":-0.12}}')),
  (4, '2024-07-01 09:03:00',
   PARSE_JSON('{"sensor_id":"S004","temp":19.8,"unit":"C","calibration":{"last_date":"2024-06-01","offset":-0.3}}')),
  (5, '2024-07-01 09:04:00',
   PARSE_JSON('{"sensor_id":"S005","temp":22.1,"unit":"C","battery":0.78,"tags":["indoor","lab"]}'));

In [0]:
select * from iot_events;

In [0]:
describe table iot_events;

In [0]:
SELECT
  event_id,
  received_at,
  payload:sensor_id          AS sensor_id,
  payload:temp               AS temp_raw,       -- raw VARIANT
  payload:temp::DOUBLE       AS temp_double,    -- cast to DOUBLE
  payload:unit::STRING       AS unit,
  payload:battery::DOUBLE    AS battery         -- NULL when field absent
FROM iot_events
ORDER BY event_id;

In [0]:
select * from iot_events;

In [0]:
SELECT
  event_id,
  payload:sensor_id::STRING               AS sensor_id,
  payload:location:lat::DOUBLE            AS latitude,
  payload:location:lon::DOUBLE            AS longitude,
  payload:calibration:last_date::STRING   AS last_calibrated,
  payload:calibration:offset::DOUBLE      AS cal_offset
FROM iot_events
ORDER BY event_id;

In [0]:
select * from iot_events;

In [0]:
SELECT
  event_id,
  payload:sensor_id::STRING  AS sensor_id,
  payload:temp::DOUBLE       AS temp
FROM  iot_events
WHERE payload:unit::STRING = 'C'
ORDER BY event_id;